# Time Lagging 示例（格点）

本 notebook 演示 **时间滞后集合**：把不同起报、但有效时间相同的预报成员，拼成一个更大的集合。

**本例设定**

| 输入 | 起报 `time` | 时效 `dtime` | 有效时间 |
| --- | --- | --- | --- |
| `cycle_00.nc` | 00 时 | 12 h | 当日 12 时 |
| `cycle_06.nc` | 06 时 | 6 h | 当日 12 时 |

合并后：起报统一到最新的 06 时，时效改成 6 h，成员数由 2+2 变成 4。  
这一步只做集合扩充，**不做**阈值概率转换。更完整说明见同目录 `time_lagging.md`。


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import meteva_base
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
repo = cwd if (cwd / "time_lagging").is_dir() else cwd.parent.parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from time_lagging.src.time_lagging import time_lag

plt.rcParams["axes.unicode_minus"] = False

testdata = repo / "time_lagging" / "testdata"
early = meteva_base.read_griddata_from_nc(str(testdata / "cycle_00.nc"))
late = meteva_base.read_griddata_from_nc(str(testdata / "cycle_06.nc"))
print("early", early.time.values, early.dtime.values, early.attrs["dtime_units"], dict(early.sizes))
print("late ", late.time.values, late.dtime.values, dict(late.sizes))

## 读数与绘图约定

- 不同起报时次必须是**不同文件/不同对象**：一份六维格点不能同时承载“00Z+12h”和“06Z+6h”
- `dtime_units` 取 `hour` 或 `minute`，有效时间 = `time + dtime`
- 下面的 `shared_panel` 会把多张成员场合用一套色标，便于跨起报比较量级


In [ ]:
def shared_panel(fields, titles, cmap="RdYlBu_r", units="degC", title=None, ncols=None, note=None):
    """画多场并共用色标；note 会 print 在图片正上方。"""
    if note:
        print(note)
    arrays = [np.asarray(item.squeeze(), dtype=float) for item in fields]
    finite = np.concatenate([arr.ravel() for arr in arrays])
    finite = finite[np.isfinite(finite)]
    vmin, vmax = float(finite.min()), float(finite.max())
    if np.isclose(vmin, vmax):
        vmax = vmin + 1.0
    n = len(fields)
    ncols = ncols or min(n, 4)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.0 * ncols, 3.4 * nrows), squeeze=False)
    last_im = None
    for index, (field, subtitle) in enumerate(zip(fields, titles)):
        ax = axes[index // ncols][index % ncols]
        plot = field.squeeze()
        last_im = ax.pcolormesh(
            plot.lon, plot.lat, plot, shading="auto", cmap=cmap, vmin=vmin, vmax=vmax
        )
        ax.set_title(subtitle)
        ax.set_xlabel("lon")
        ax.set_ylabel("lat")
        ax.set_aspect("equal", adjustable="box")
    for index in range(n, nrows * ncols):
        axes[index // ncols][index % ncols].axis("off")
    cbar = fig.colorbar(last_im, ax=axes.ravel().tolist(), fraction=0.046, pad=0.04)
    cbar.set_label(units)
    if title:
        fig.suptitle(title)
    print(f"色标范围: vmin={vmin:.3f}, vmax={vmax:.3f} ({units})")
    plt.show()
    plt.close("all")
    return vmin, vmax


## 合并前：两个起报时次的成员场

四张图从左到右是：00Z 两个成员 + 06Z 两个成员，**共用同一色标**。

观察要点：

- 同一有效时刻、不同起报的成员，数值可以不同，这正是时间滞后要保留的多样性
- 空间结构应能对齐（经纬度、层次一致），否则后面合并会报错


In [ ]:
early_members = [early.sel(member=m).squeeze(drop=True) for m in early.member.values]
late_members = [late.sel(member=m).squeeze(drop=True) for m in late.member.values]

shared_panel(
    early_members + late_members,
    [f"00Z/{m}" for m in early.member.values] + [f"06Z/{m}" for m in late.member.values],
    title="members before lagging (shared colorbar)",
    ncols=4,
    note="【图】合并前：00Z 两成员 + 06Z 两成员，同一有效时刻、同一色标。跨起报差异将被时间滞后保留进大集合。",
)


## 时间滞后合并

`time_lag(early, late)` 做的事情：

1. 检查各输入的有效时间集合是否一致
2. 把所有起报时间改写为**最新起报**（这里是 06 时）
3. 重算 `dtime`，使有效时间不变（00Z 的 12h → 06Z 的 6h）
4. 成员编号按输入顺序从 0 重新编排，再沿 `member` 维拼接

打印结果里应看到：起报 06 时、时效 6、成员 `0..3`，且各成员空间平均约为 `1,2,3,4`。


In [ ]:
print("开始时间滞后合并：要求有效时间一致，起报统一到最新时次并重算时效。")
lagged = time_lag(early, late)
print("合并结果 → 起报:", lagged.time.values, " 时效:", lagged.dtime.values)
print("成员编号:", list(lagged.member.values))
print("各成员空间平均:", lagged.mean(dim=("lat", "lon")).values.reshape(-1))


## 合并后成员场（同一图例）

合并后的 4 个成员再次用统一色标画出，并附上集合平均与成员标准差：

- 平均场：时间滞后集合的中心估计
- 标准差：成员之间的离散程度（集合越大，通常越能刻画不确定性）

若色标范围内能同时看见来自 00Z 与 06Z 的信息，说明合并成功保留了跨起报差异。


In [ ]:
lagged_members = [lagged.sel(member=m).squeeze(drop=True) for m in lagged.member.values]
shared_panel(
    lagged_members,
    [f"lagged member {m}" for m in lagged.member.values],
    title=f"after lagging  init={pd.Timestamp(lagged.time.values[0])}  lead={int(lagged.dtime.values[0])}h",
    ncols=4,
    note="【图】合并后 4 个成员（起报已统一到 06Z，时效 6h），色标统一，可同时看到来自两个起报的信息。",
)

ens_mean = lagged.mean("member").squeeze(drop=True)
ens_std = lagged.std("member").squeeze(drop=True)
shared_panel(
    [ens_mean, ens_std],
    ["lagged mean", "member std"],
    title="lagged ensemble mean / std",
    ncols=2,
    note="【图】左：滞后集合平均；右：成员标准差（离散度）。",
)


## 站点路径核对

同一套 `time_lag` 也适用于 meteva 站点表（`level,time,dtime,id,lon,lat` + `member`）。

这里用 CSV 示例再跑一遍，确认：

- 起报/时效规则与格点一致
- 站号身份保持不变
- 折线图上每个站的 4 个成员数值符合预期

格点与站点只是载体不同，时间滞后逻辑相同。


In [ ]:
early_station = meteva_base.read_stadata_from_csv(str(testdata / "cycle_00.csv"), drop_same_id=False)
late_station = meteva_base.read_stadata_from_csv(str(testdata / "cycle_06.csv"), drop_same_id=False)
lagged_station = time_lag(early_station, late_station)
print("【表】站点时间滞后结果预览（规则与格点相同）：")
print(lagged_station.head(8))

pivot = lagged_station.pivot_table(index="id", columns="member", values="air_temperature")
print("【表】站号 × 成员：")
print(pivot)

print("【图】每个站的 4 个时间滞后成员曲线。")
plt.close("all")
fig, ax = plt.subplots(figsize=(6.5, 3.8))
x = pivot.columns.astype(float).to_numpy()
for station_id, row in pivot.iterrows():
    ax.plot(x, row.to_numpy(), marker="o", label=str(int(station_id)))
ax.set_xlabel("member")
ax.set_ylabel("air_temperature")
ax.set_title("station time-lagged members")
ax.legend(title="station id")
plt.show()
plt.close("all")
